# Объединение месячных CSV и загрузка в DRP

Эта тетрадка объединяет несколько CSV с одинаковой структурой и загружает результат в DRP (Greenplum).

Запуск:
1. Указать пути к файлам в параметрах.
2. Проверить схему и результат объединения.
3. Ввести логин/пароль DRP.
4. Запустить ячейку загрузки.

In [ ]:
import pandas as pd
from pathlib import Path
from getpass import getpass
from rail_connectors.connection import connect

## 1) Параметры

In [ ]:
# Пути к CSV-файлам, которые нужно объединить
base_dir = '/home/jovyan/documents/Equaring/Проверки/Фин.скрипт'
csv_files = [
    f"{base_dir}/datamart_01_month_acquiring_2026_05.csv",
    f"{base_dir}/datamart_02_month_acquiring_2026_05.csv",
    f"{base_dir}/datamart_03_month_acquiring_2026_05.csv",
]

# DRP target
drp_schema = 'sbx_da'
drp_table = 'tmp_datamart_month_acquiring_2026_05'
drp_mode = 'replace'  # replace / append

# Удалить полные дубли строк после объединения
drop_full_duplicates = True

## 2) Проверка файлов и схемы

In [ ]:
for p in csv_files:
    path_obj = Path(p)
    print(path_obj, 'OK' if path_obj.exists() else 'NOT FOUND')

heads = []
for p in csv_files:
    h = pd.read_csv(p, nrows=0)
    heads.append((p, list(h.columns)))

base_cols = heads[0][1]
mismatch = []
for p, cols in heads[1:]:
    if cols != base_cols:
        mismatch.append(p)

if mismatch:
    print('ВНИМАНИЕ: различается схема колонок в файлах:')
    for p in mismatch:
        print('-', p)
else:
    print('Схема колонок совпадает во всех файлах.')

print('Колонок:', len(base_cols))
base_cols[:20]

## 3) Объединение

In [ ]:
parts = [pd.read_csv(p, dtype=object) for p in csv_files]
merged_df = pd.concat(parts, ignore_index=True)

rows_before = len(merged_df)
if drop_full_duplicates:
    merged_df = merged_df.drop_duplicates().reset_index(drop=True)
rows_after = len(merged_df)

if 'report_month' in merged_df.columns:
    merged_df['report_month'] = pd.to_datetime(merged_df['report_month'], errors='coerce')
if 'snapshot_month_start' in merged_df.columns:
    merged_df['snapshot_month_start'] = pd.to_datetime(merged_df['snapshot_month_start'], errors='coerce')

print('rows_before =', rows_before)
print('rows_after  =', rows_after)
print('cols        =', len(merged_df.columns))
merged_df.head(5)

## 4) Загрузка в DRP

После записи сразу выполняем проверку: количество строк, уникальных ИНН и sample из таблицы.

In [ ]:
drp_user = input('DRP user: ').strip()
drp_password = getpass('DRP password: ')

target_fq = f'{drp_schema}.{drp_table}'
print('target =', target_fq)
print('mode   =', drp_mode)
print('rows   =', len(merged_df))

drp_conn = connect(
    to='DRP',
    user_params={
        'user_name': drp_user,
        'password': drp_password,
    }
)

with drp_conn:
    drp_conn.write(
        table=drp_table,
        schema=drp_schema,
        df=merged_df,
        mode=drp_mode,
    )

print('DRP upload finished:', target_fq)

sql_cnt = f"""
select
  count(*) as row_cnt,
  count(distinct inn) as inn_cnt
from {target_fq}
"""

sql_sample = f"""
select *
from {target_fq}
limit 10
"""

with drp_conn:
    cnt_df = drp_conn.fetch(sql_cnt)
    sample_df = drp_conn.fetch(sql_sample)

print('Проверка загрузки в DRP:')
display(cnt_df)
print('Sample rows:')
display(sample_df)